In [ ]:
# etapa 2 - concatenar os dados de criminalidade
import pandas as pd 

# para guardar os dataframes
lista_dfs_crimes = []

anos = [2021, 2022, 2023, 2024, 2025, 2026]
for ano in anos:
    caminho = f'Data/dado_criminal/SPJ_{ano}.csv'
    df_temp = pd.read_csv(caminho, sep=';', encoding='latin-1', low_memory=False)
    lista_dfs_crimes.append(df_temp)

df_crimes_rs = pd.concat(lista_dfs_crimes, ignore_index=True)


# drop das colunas vazias, e também toda coluna q tiver como unnamed no nome as deleta
colunas_vazias = [col for col in df_crimes_rs.columns if 'Unnamed' in str(col)]
df_crimes_rs.drop(columns=colunas_vazias, inplace=True)


# renomeado as colunas para facilitar 
df_crimes_rs.rename(columns={
    "Sequência": "id",
    "Data Fato": "data",
    "Hora Fato": "hora",
    "Grupo Fato": "grupo_crime",
    "Tipo Enquadramento": "crime",
    "Tipo Fato": "status",
    "Municipio Fato": "municipio",
    "Local Fato": "local",
    "Bairro": "bairro",
    "Quantidade Vítimas": "qtd_vitimas",
    "Idade Vítima": "idade_vitima",
    "Sexo Vítima": "sexo_vitima",
    "Cor Vítima": "cor_vitima"
}, inplace=True)

# drop da coluna ... que sobrou e ta no csv
if '...' in df_crimes_rs.columns:
    df_crimes_rs.drop(columns=['...'], inplace=True)

# visualização
print("Tamanho do DataSet", df_crimes_rs.shape)
df_crimes_rs.info()

# padronização de municipio
df_crimes_rs["municipio"] = df_crimes_rs["municipio"].str.strip().str.lower()

#filtro de passo fundo
df_crimes_pf = df_crimes_rs[df_crimes_rs["municipio"] == "passo fundo"]

print(df_crimes_pf["municipio"].unique())
print(df_crimes_pf.shape)


Tamanho do DataSet (3218268, 13)
<class 'pandas.DataFrame'>
RangeIndex: 3218268 entries, 0 to 3218267
Data columns (total 13 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            object 
 1   data          str    
 2   hora          str    
 3   grupo_crime   str    
 4   crime         str    
 5   status        str    
 6   municipio     str    
 7   local         str    
 8   bairro        str    
 9   qtd_vitimas   float64
 10  idade_vitima  float64
 11  sexo_vitima   str    
 12  cor_vitima    str    
dtypes: float64(2), object(1), str(10)
memory usage: 319.2+ MB
<StringArray>
['passo fundo']
Length: 1, dtype: str
(67274, 13)


In [ ]:
# etapa 3

# idade_vitima preenche com média
df_crimes_pf["idade_vitima"] = df_crimes_pf["idade_vitima"].fillna(df_crimes_pf["idade_vitima"].mean())

#sexo_vitima preenche com ignorado
df_crimes_pf["sexo_vitima"] = df_crimes_pf["sexo_vitima"].fillna("ignorado")

# cor_vitima preenche com ignorado
df_crimes_pf["cor_vitima"] = df_crimes_pf["cor_vitima"].fillna("ignorado")

# bairro preenche como desconhecido
df_crimes_pf["bairro"] = df_crimes_pf["bairro"].fillna("desconhecido")


#remover duplicatas
df_crimes_pf.drop_duplicates(inplace=True)


#padronização de texto
colunas_texto = ["grupo_crime", "crime", "status", "bairro"]

for col in colunas_texto:
    df_crimes_pf[col] = df_crimes_pf[col].str.lower().str.strip()

# padronização da data
df_crimes_pf["data"] = pd.to_datetime(df_crimes_pf["data"], format="%d/%m/%Y")


# verificação
print("valores nulos pós tratamento: ")
print(df_crimes_pf.isna().sum())

print("\nShape final:")
print(df_crimes_pf.shape)


valores nulos pós tratamento: 
id              0
data            0
hora            0
grupo_crime     0
crime           0
status          0
municipio       0
local           0
bairro          0
qtd_vitimas     0
idade_vitima    0
sexo_vitima     0
cor_vitima      0
dtype: int64

Shape final:
(67274, 13)


In [27]:
# etapa 4
# agrupar por data
df_crimes_pf["data"] = pd.to_datetime(df_crimes_pf["data"], errors="coerce")
df_crimes_diario = df_crimes_pf.groupby("data").size().reset_index(name="qtd_crimes")

# import de dados
lista_clima = []

for ano in anos:
    caminho = f'Data/dado_clima/INMET_{ano}.CSV'
    
    df_temp = pd.read_csv(caminho, sep=';', encoding='latin-1', skiprows=8) # =8 para pular cabeçalho inicial
    lista_clima.append(df_temp)

df_clima = pd.concat(lista_clima, ignore_index=True)

# nomes simplificados para as principais colunas
df_clima.rename(columns={
    "Data": "data",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)": "precipitacao",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)": "temperatura"
}, inplace=True)


# converte a data
df_clima["data"] = pd.to_datetime(df_clima["data"], format="%Y/%m/%d")

#converter para numero pq ta str
df_clima["temperatura"] = df_clima["temperatura"].str.replace(",", ".")
df_clima["temperatura"] = pd.to_numeric(df_clima["temperatura"], errors="coerce")

df_clima["precipitacao"] = df_clima["precipitacao"].str.replace(",", ".")
df_clima["precipitacao"] = pd.to_numeric(df_clima["precipitacao"], errors="coerce")

# agrupa por dia
df_clima_diario = df_clima.groupby("data").agg({
    "precipitacao": "sum",
    "temperatura": "mean"
}).reset_index()


# merge dos dados
df_final = pd.merge(df_crimes_diario, df_clima_diario, on="data", how="inner")
df_final["temperatura"] = df_final["temperatura"].fillna(df_final["temperatura"].mean())

#validacao
print("Shape do dataset final:")
print(df_final.shape)

print("\nValores nulos:")
print(df_final.isna().sum())





Shape do dataset final:
(618, 4)

Valores nulos:
data            0
qtd_crimes      0
precipitacao    0
temperatura     0
dtype: int64
